In [0]:
CATALOG_NAME = "subscription_pipeline"
BRONZE_SCHEMA = "bronze"

# Create the catalog if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")

# Create the bronze schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}")

# 2. List your 6 Fivetran tables (Update these names to match what Fivetran created)
# Example: Fivetran usually puts them in a schema like 'fivetran_raw'
fivetran_source_schema = "analytics.azure_blob_storage"
tables_to_process = ["country_master", "customer", "employee", "fx_rate", "opportunity", "product"]

from pyspark.sql.functions import current_timestamp, lit

for table in tables_to_process:
    print(f"Processing {table} into Bronze Staging...")
    
    # Read the data from where Fivetran landed it
    df_raw = spark.read.table(f"{fivetran_source_schema}.{table}")
    
    # 3. Add 'Staging' Metadata (Audit Columns)
    df_stg = df_raw.withColumn("load_timestamp", current_timestamp()) \
                   .withColumn("source_system", lit("fivetran_blob"))
    
    # 4. Write to the Bronze Layer with 'stg_' prefix
    # Use 'overwrite' to keep it fresh or 'append' if you want history
    target_table_name = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.stg_{table}"
    
    df_stg.write.format("delta") \
          .mode("overwrite") \
          .option("overwriteSchema", "true") \
          .saveAsTable(target_table_name)

print("All 6 tables are now standardized in the Bronze (Staging) layer.")